<img src="./assets/课程封面.png"  width="1024px">  

# 大数据（向量数据库） & RAG
**说明**：将从大模型的局限性出发，代码实现一个完整的 RAG 系统，并深入探讨切分策略、重排序（Rerank）、混合检索、多模态表格处理等进阶技巧。

## 💡 这节课会带给你

1. **核心原理**：如何用你的向量数据库（垂域数据）补充 LLM 的能力（解决幻觉与时效性）。
    
2. **实战构建**：如何从零构建你的向量（垂域）知识库。
    
3. **系统架构**：搭建一套完整 RAG 系统需要哪些模块（离线处理 vs 在线检索）。
    
4. **进阶技巧**：搭建 RAG 系统时企业级落地的有用技巧（重排序、混合检索、表格处理）。
    

* * *

## 一、什么是检索增强的生成模型（RAG）

### 1.1 大模型目前固有的局限性

我们在讨论大模型（LLM）时，通常指的是**预训练模型**（Pre-trained Models），如 Qwen, Llama, GPT-4 等。 预训练模型的本质可以概括为：**特定的算法结构 + 海量的数据**。

虽然它们非常强大，但在企业落地时存在以下显著的局限性：

1. **知识不是实时的（时效性问题）**：模型的知识截止于训练数据的收集时间。例如，Llama 2 或 GPT-4 的早期版本无法回答昨天刚发生的新闻。
    
2. **缺乏私有数据（私有化问题）**：公有大模型没有经过企业内部数据或个人私有数据的训练，因此无法回答与内部业务、私有文档相关的问题。
    
3. **幻觉问题（Hallucination）**：当模型不知道答案时，它倾向于根据概率生成看似通顺但完全错误的内容（一本正经地胡说八道）。
    

**解决方案对比：微调 (Fine-tuning) vs RAG**

- **微调**：将新知识“训练”进模型的大脑。
    
    - *缺点*：成本高、周期长、不仅难以做到实时更新，而且对于精准的事实性知识（如具体的订单号、政策条款），微调的效果往往不如预期，仍然可能产生幻觉。
        
- **RAG**：不给模型“动手术”，而是给它外挂一个知识库。
    
    - *优点*：成本低、数据更新快（毫秒级）、答案可追溯。
        

### 1.2 检索增强生成

**RAG (Retrieval Augmented Generation)** 顾名思义，由三个核心部分组成：

- **R (Retrieval)**：检索
    
- **A (Augmented)**：增强
    
- **G (Generation)**：生成
    

它的核心逻辑是：**通过检索的方法来增强生成模型的能力**。

<video src="./assets/RAG.mp4" controls="controls" width=1024px style="margin-left: 0px"></video>

<div class="alert alert-success"> <b>💡 形象类比：</b>

你可以把这个过程想象成<b>“开卷考试”</b>。

传统的 LLM 问答是“闭卷考试”，模型只能靠脑子里的记忆（预训练参数）回答。

RAG 则是“开卷考试”：允许模型先“翻书”（在向量数据库中检索相关文档），找到参考资料后，再结合问题进行回答。这样既准确，又能有据可依。 </div>

> **【🔍 深度解析 & 关联性解读】**
> 
> - **逻辑关联**：这部分奠定了全篇的基调。后续的所有代码（PDF解析、向量化、Prompt构建）都是为了解决这里提出的三个问题（时效性、私有数据、幻觉）。
>     
> - **RAG vs 微调**：这是一个高频面试题。这里的核心区别在于：微调改变了模型的**参数（记忆）**，而 RAG 改变了模型的**输入（上下文）**。后续代码演示的正是如何构造这个“输入”。
>     



* * *

## 二、RAG系统的基本搭建流程

RAG 的架构在逻辑上严格分为两个阶段：**离线阶段（数据入库）** 和 **在线阶段（检索生成）**。

<img src="./assets/rag.png" style="margin-left: 0px; background-color: white; padding: 10px;" width="1024px">


**详细流程拆解：**

1. **离线阶段 (Offline)**：
    
    - **文档加载**：读取 PDF、Word、Excel 等私有数据。
        
    - **文档切分 (Splitting)**：将长文档按一定规则（如换行符、语义完整性）切割成小的文本片段（Chunks）。目的是适应 Embedding 模型的输入窗口限制，并提高检索的颗粒度。
        
    - **向量化 (Embedding)**：使用 Embedding 模型将文本片段转为向量（一串数字）。
        
    - **灌库**：将向量及其对应的原始文本存入**向量数据库**。
        
2. **在线阶段 (Online)**：
    
    - **用户提问 (Query)**。
        
    - **查询转换**：使用**同一个** Embedding 模型将用户的问题也转为向量。
        
    - **检索 (Retrieval)**：在向量数据库中计算“问题向量”与“文档向量”的相似度（如余弦相似度），召回 Top-K 个最相关的片段。
        
    - **增强 (Augmented)**：将检索到的文档片段（Context）和用户问题（Query）填入预设的 Prompt 模板。
        
    - **生成 (Generation)**：将填充好的 Prompt 发送给 LLM，生成最终回复。
        

> **【🔍 深度解析 & 关联性解读】**
> 
> - **架构解读**：这张图是 RAG 的灵魂。请注意图中**Embedding**这个模块出现了两次（离线一次，在线一次）。这非常关键：**必须使用同一个模型**将文档和问题映射到同一个向量空间，否则距离计算没有意义。
>     
> - **数据流向**：后续的代码顺序完全按照这个流程进行：先写文档加载器 -> 再写向量化 -> 再写存库 -> 最后写检索和生成。
>     

### 2.1 文档的加载与切割

首先，我们需要处理源文件。这里以 PDF 文档为例。我们将不使用高级框架（如 LangChain），而是使用底层的 `pdfminer` 库，以便大家理解底层原理。


In [ ]:
!pip install --upgrade openai


Python

In [2]:
# 安装 pdf 解析库
!pip install pdfminer.six


下面的代码定义了一个函数，用于从 PDF 中提取文本，并进行简单的段落清洗。


In [1]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextContainer

def extract_text_from_pdf(filename, page_numbers=None, min_line_length=1):  # 从 PDF 中提取文本段落
    '''从 PDF 文件中（按指定页码）提取文字'''
    paragraphs = []
    buffer = ''
    full_text = ''
    # 提取全部文本
    for i, page_layout in enumerate(extract_pages(filename)):  # 逐页解析 PDF 内容
        # 如果指定了页码范围，跳过范围外的页
        if page_numbers is not None and i not in page_numbers:
            continue
        for element in page_layout:
            if isinstance(element, LTTextContainer):
                full_text += element.get_text() + '\n'
    # 按空行分隔，将文本重新组织成段落
    lines = full_text.split('\n')
    for text in lines:
        if len(text) >= min_line_length:
            # 简单的拼接逻辑，处理断行，如果行尾是连字符则去掉
            buffer += (' '+text) if not text.endswith('-') else text.strip('-')
        elif buffer:
            paragraphs.append(buffer)
            buffer = ''
    if buffer:
        paragraphs.append(buffer)
    return paragraphs


**测试提取内容：** 我们加载一份示例 PDF 文档（Llama 2 的论文），并查看提取出的前几段内容。这里使用的是简单的换行符切割，后续进阶部分我们会讲更高级的切割。


📁 **数据集：Llama 2 论文 PDF（示例文档）**

- 文件：`llama2.pdf`（Meta 发布的 Llama 2 论文原文，本地示例文档）
- 用途：作为 RAG 离线阶段的"待检索语料"，演示 PDF 文本提取 → 向量化入库 → 检索问答全流程
- 规模：约 77 页论文正文（含摘要、模型架构、训练细节、评测结果等章节）
- 处理方式：先用 `pdfminer` 逐页提取文字，再按段落清洗；后续章节还会基于它做文本切分（Chunk）与表格图片抽取（第 8 页）


> **【🔍 深度解析 & 关联性解读】**
> 
> - **为什么用** `pdfminer`？：相比于 LangChain 等封装好的库，直接用 `pdfminer` 能让我们看到最原始的数据清洗过程。
>     
> - **代码细节** `buffer += ...`：这一段逻辑是为了解决 PDF 解析中常见的问题——**硬换行**。PDF 中一行文字结束并不代表段落结束。代码尝试将原本属于同一段落的断行拼接起来，这对后续的 RAG 检索至关重要，因为**语义完整的段落才能生成高质量的向量**。如果段落被打断，语义就会破碎，导致检索失败。
>     

### 2.2 LLM 接口封装

接下来，我们需要封装调用大模型的接口。这里使用 OpenAI 的官方 SDK。

**经验分享：环境变量的管理** 在企业开发中，我们通常有两种方式管理 API Key：

1. **系统环境变量**：在 Linux/Windows 系统层面配置。缺点是跨平台不方便，容易混淆。
    
2. `.env` 文件（推荐）：在项目根目录下创建一个 `.env` 文件，配合 `python-dotenv` 库读取。这种方式配置灵活，且不会把 Key 硬编码在代码里，更加安全。
    

In [4]:
paragraphs = extract_text_from_pdf("llama2.pdf", min_line_length=10)  # 从 PDF 中提取文本段落

for para in paragraphs[:4]:
    print(para+"\n")

 Llama 2: Open Foundation and Fine-Tuned Chat Models

 Hugo Touvron∗ Louis Martin† Kevin Stone† Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic Sergey Edu

In [5]:
!pip install -U python-dotenv

  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.1
    Uninstalling python-dotenv-1.2.1:
      Successfully uninstalled python-dotenv-1.2.1



> **【🔍 深度解析 & 关联性解读】**
> 
> - `temperature=0` 的重要性：这里设置温度为 0 是 RAG 系统的一个关键最佳实践。
>     
>     - **关联性**：在 1.1 节我们提到了“幻觉问题”。RAG 的目的是基于检索到的事实回答。如果温度过高，模型会倾向于自由发挥，即使我们提供了参考资料，它也可能编造内容。设置为 0 能最大程度约束模型，使其忠实于 prompt 中的上下文。
>         

### 2.3 Prompt 模板

这是 RAG 中“**A (Augmented)**”的关键步骤。我们需要构建一个模板，告诉大模型：“请根据我给你的已知信息（Context）来回答用户的问题（Query）”。

**Prompt 设计要点**：

- 明确角色：你是一个问答机器人。
    
- 明确任务：基于已知信息回答。
    
- **抑制幻觉**：明确告诉模型，如果已知信息里没有答案，就直接回答“我不知道”，不要编造。
    


In [6]:
from openai import OpenAI
import os

# 加载环境变量
# 建议在项目根目录下创建一个 .env 文件，内容为：OPENAI_API_KEY=sk-xxxx
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), verbose=True)  # 读取本地 .env 文件

client = OpenAI()  # 创建 OpenAI 客户端（用于 Embedding 与问答）

def get_completion(prompt, model="gpt-4o"):  # 定义大模型问答函数（调用 LLM 生成回答）
    '''封装 openai 接口'''
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(  # 调用 LLM 接口生成回答
        model=model,
        messages=messages,
        # temperature 控制输出的随机性。
        # 在 RAG 场景中，我们希望模型基于事实回答，因此通常设置为 0，减少模型“发挥”的空间。
        # 如果是创意写作场景，可以设置为 0.8 或更高。
        temperature=0,  
    )
    return response.choices[0].message.content


> **【🔍 深度解析 & 关联性解读】**
> 
> - **Prompt 中的** `{context}`：这是一个占位符。在后续的**在线处理阶段**，我们会将从向量数据库检索出来的 Top-K 个文档片段拼接到这里。
>     
> - **逻辑闭环**：这个 Template 对应了架构图中的 "Prompt" 环节。它是连接“检索器（Retriever）”和“生成器（Generator/LLM）”的桥梁。如果没有这个步骤，检索出来的知识就无法被 LLM 利用。
>     

* * *

## 三、向量检索

在 RAG 中，为了让计算机理解文本的语义并进行快速检索，我们需要使用**向量（Vectors）**。

### 3.1 什么是向量

向量是数学对象，在多维空间中表示一个点或方向。 对于文本而言，**Embedding（嵌入）** 技术将一段文本转化为一组 N 维的浮点数（例如 `[0.12, -0.58, 0.99, ...]`）。

<br /> <img src="./assets/vector.png" style="margin-left: 0px; background-color: white; padding: 10px;" width=800px> <br />

#### 3.1.1 文本向量的特性

1. **降维打击**：将复杂的自然语言转化为计算机擅长处理的数字数组。
    
2. **语义空间**：向量之间可以计算距离。距离越近，代表文本的**语义相似度**越高。
    
    - *举例*：“这个多少钱？”、“什么价格？”、“给我报个价”。这三句话字面上完全不同，但在向量空间中，它们的位置会非常接近（距离短）。
        

<br /> <img src="./assets/embeddings.png" style="margin-left: 0px; background-color: white; padding: 10px;" width=800px> <br />

### 3.1.2 文本向量是怎么得到的

Embedding 模型通常是基于**双塔模型**训练出来的。训练过程让语义相关的句子（正例）距离更近，不相关的句子（负例）距离更远。

<img src="./assets/sbert.png" style="margin-left: 0px" width=500px>

<div class="alert alert-info"> <b>扩展阅读：[https://www.sbert.net](https://www.sbert.net)</b>

推荐大家了解 <b>Sentence-BERT</b>，这是目前开源界非常流行的 Embedding 模型架构。 </div>


In [7]:
def build_prompt(prompt_template, **kwargs):  # 将检索结果与用户问题填入 Prompt 模板
    '''将 Prompt 模板赋值'''
    inputs = {}
    # 遍历传入的所有关键字参数 (kwargs)
    for k, v in kwargs.items():
        # 核心逻辑：处理列表类型的输入
        # 如果参数值是一个列表，且列表里都是字符串（通常是检索到的多条文档片段）
        if isinstance(v, list) and all(isinstance(elem, str) for elem in v):
            # 将列表中的多条文档用两个换行符连接成一个长字符串
            val = '\n\n'.join(v)
        else:
            # 如果不是列表（比如是用户的问题 query），直接使用原值
            val = v
        inputs[k] = val
    return prompt_template.format(**inputs)

prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。

已知信息:
{context} # 这里将填入从向量数据库检索出来的原始文档片段

用户问：
{query} # 这里填入用户的提问

如果已知信息不包含用户问题的答案，或者已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。
请不要输出已知信息中不包含的信息或答案。
请用中文回答用户问题。
"""


### 3.2 向量间的相似度计算

如何判断两个向量是否相似？主要有两种数学方法：

1. **余弦相似度 (Cosine Similarity)**：计算两个向量夹角的余弦值。
    
    - **值越大（越接近 1），越相似**。
        
    - 如果夹角为 0，余弦值为 1，表示完全相同。
        
    - 这是 NLP 中最常用的方法，因为它主要关注方向（语义）而非大小。
        
2. **欧氏距离 (Euclidean Distance / L2)**：计算两个点之间的直线距离。
    
    - **值越小，越相似**。
        

<img src="./assets/sim.png" style="margin-left: 0px; background-color: white; padding: 10px;" width=500px>

我们编写代码来封装 OpenAI 的 Embedding 接口，并实现相似度计算函数。

**关于模型选择的建议**：

- **OpenAI** `text-embedding-ada-002` / `3-large`：综合能力最强，特别是**跨语言能力**（中文问英文文档）非常出色，维度通常为 1536 或 3072。
    
- **开源模型 (HuggingFace)**：如果需要私有化部署，可以去 MTEB 榜单找排名靠前的模型（如 BGE, E5 等）。但很多开源模型对跨语言支持不如 OpenAI。
    


In [8]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
# 注意：要让 get_embeddings 函数正常工作，您需要在外部定义 client 对象，
# 通常是导入 openai 库并初始化：from openai import OpenAI; client = OpenAI()

def cos_sim(a, b):
    '''
    函数功能：计算两个向量之间的余弦相似度。
    相似度度量：值越大（接近 1）表示向量方向越接近，文本内容越相似。
    公式：(a · b) / (||a|| * ||b||)
    参数：
        a (array-like): 第一个向量。
        b (array-like): 第二个向量。
    返回值：
        float: 余弦相似度值。
    '''
    # 向量点积 (a · b) / 向量 a 的 L2 范数 (||a||) * 向量 b 的 L2 范数 (||b||)
    return dot(a, b) / (norm(a) * norm(b))

def l2(a, b):
    '''
    函数功能：计算两个向量之间的欧氏距离（L2 范数距离）。
    距离度量：值越小（接近 0）表示向量在空间中的距离越近，文本内容越相似。
    公式：sqrt(sum((a_i - b_i)^2))，即两个向量差的 L2 范数。
    参数：
        a (array-like): 第一个向量。
        b (array-like): 第二个向量。
    返回值：
        float: 欧氏距离值。
    '''
    # 1. 将输入转换为 NumPy 数组并计算向量差 (a - b)
    x = np.asarray(a) - np.asarray(b)
    # 2. 计算差向量的 L2 范数（即欧氏距离）
    return norm(x)

def get_embeddings(texts, model="text-embedding-ada-002", dimensions=None):  # 将文本转为向量
    '''
    函数功能：封装调用 OpenAI 的 Embedding 模型接口，生成文本的向量嵌入。
    
    参数：
        texts (str or list[str]): 需要生成嵌入的文本或文本列表。
        model (str): 指定用于生成嵌入的模型名称，默认为 OpenAI 的经典模型。
        dimensions (int, optional): 可选参数，用于指定返回向量的维度。
                                    对于 text-embedding-ada-002，默认维度为 1536。

    返回值：
        list[list[float]]: 包含每个输入文本对应嵌入向量的列表。
    '''
    # OpenAI 官方文档提示：text-embedding-ada-002 默认维度为 1536，
    # 如果明确使用了该模型，则不需要设置 dimensions 参数。
    if model == "text-embedding-ada-002":
        dimensions = None
        
    # 根据是否需要指定维度来构造 API 调用
    if dimensions:
        # 调用 API 接口，指定输入文本、模型和所需的维度
        data = client.embeddings.create(  # 调用 Embedding 模型生成向量
            input=texts, model=model, dimensions=dimensions).data
    else:
        # 调用 API 接口，只指定输入文本和模型（使用模型默认维度）
        data = client.embeddings.create(input=texts, model=model).data  # 调用 Embedding 模型生成向量
        
    # 从返回的数据对象中提取出实际的嵌入向量（embedding 字段）并以列表形式返回
    return [x.embedding for x in data]


**实战测试：** 我们来验证一下 Embedding 的效果。我们定义一个查询词“国际争端”，和几句不同的话，看看计算出来的相似度是否符合人类直觉。



> **结果分析**：你会发现前三句话的余弦相似度得分（例如 0.8+）明显高于后两句（例如 0.7 左右）。只要得分是“挨着”且区分度明显的，就说明模型好用。

> **【🔍 深度解析 & 关联性解读】**
> 
> - `get_embeddings` 的作用：这个函数是 RAG 系统中最底层的“翻译官”，它负责将人类语言翻译成机器语言（向量）。
>     
> - **测试的意义**：在正式灌入数据库之前，进行这样的测试非常重要。它能验证你选择的 Embedding 模型是否能正确理解“国际争端”这个语义。如果你发现毫不相关的句子得分也很高，说明模型质量差，或者不适合当前的领域。
>     

### 3.3 向量数据库

向量数据库（Vector Store）是专门为向量检索设计的中间件。它可以高效地存储海量向量，并支持快速的最近邻搜索（ANN）。

- **为什么需要它？** 随着文档增多，简单的 Numpy 遍历计算无法满足性能需求（IO 瓶颈），且我们需要数据持久化。
    
- **澄清**：向量数据库**本身不生成向量**，它只负责**存储**和**检索**。向量是由 Embedding 模型生成的。
    

#### 3.3.1 实战：使用 Chroma 向量数据库

本课程使用 **ChromaDB**，因为它轻量且易于在本地运行。

官方文档：[https://docs.trychroma.com/docs/overview/introduction](https://docs.trychroma.com/docs/overview/introduction)

<img src="./assets/chroma.svg" style="margin-left: 0px" width=800px>

首先安装并准备数据：


In [9]:
test_query = ["测试文本"]
vec = get_embeddings(test_query)[0]  # 将文本转为向量
print(f"Total dimension: {len(vec)}") # 应该输出 1536
print(f"First 10 elements: {vec[:10]}")


Total dimension: 1536
First 10 elements: [-0.007280503865331411, -0.006169671658426523, -0.010576579719781876, 0.001448634546250105, -0.010707695037126541, 0.02919485792517662, -0.019725468009710312, 0.0053902678191661835, -0.016957491636276245, -0.01203340943902731]


In [10]:
# 实战测试
query = "国际争端"
# query = "global conflicts" # 跨语言测试也可以，OpenAI 模型跨语言能力很强

documents = [
    "联合国就苏丹达尔富尔地区大规模暴力事件发出警告", # 强相关
    "土耳其、芬兰、瑞典与北约代表将继续就瑞典“入约”问题进行谈判", # 相关
    "日本岐阜市陆上自卫队射击场内发生枪击事件 3人受伤", # 弱相关/军事相关
    "国家游泳中心（水立方）：恢复游泳、嬉水乐园等水上项目运营", # 不相关
    "我国首次在空间站开展舱外辐射生物学暴露实验", # 不相关
]

query_vec = get_embeddings([query])[0]  # 将文本转为向量
doc_vecs = get_embeddings(documents)  # 将文本转为向量

print("Query与自己的余弦距离: {:.2f}".format(cos_sim(query_vec, query_vec)))
print("Query与Documents的余弦距离:")
for vec in doc_vecs:
    print(cos_sim(query_vec, vec))

print()

print("Query与自己的欧氏距离: {:.2f}".format(l2(query_vec, query_vec)))
print("Query与Documents的欧氏距离:")
for vec in doc_vecs:
    print(l2(query_vec, vec))

Query与自己的余弦距离: 1.00
Query与Documents的余弦距离:
0.8218706620454886
0.8293702601480339
0.7977241623434768
0.7668417592948099
0.7932053080963863

Query与自己的欧氏距离: 0.00
Query与Documents的欧氏距离:
0.5968741071718394
0.5841741840972877
0.6360437776854144
0.682873678050407
0.6431091648124254


In [ ]:
!pip install chromadb

In [13]:

# 为了演示方便，我们只取 PDF 的第2、3页（第一章内容）
paragraphs = extract_text_from_pdf(  # 从 PDF 中提取文本段落
    "llama2.pdf",
    page_numbers=[2, 3],
    min_line_length=10
)


### 封装一个向量数据库连接器：


In [14]:
import chromadb  # 向量数据库（Chromadb）
from chromadb.config import Settings

class MyVectorDBConnector:
    def __init__(self, collection_name, embedding_fn):
        # 1. 内存模式 (数据存在内存，重启即失，仅供测试)
        chroma_client = chromadb.Client(Settings(allow_reset=True))
        
        # 2. 持久化模式 (推荐企业开发使用，数据落盘)
        # chroma_client = chromadb.PersistentClient(path="./chroma_db")

        # 注意：为了演示方便，我这里使用了 reset() 清空数据。
        # 在生产环境中，千万不要调用 reset()，这是不可逆的操作！
        chroma_client.reset()

        # 创建一个 collection (类似于关系型数据库中的 Table)
        self.collection = chroma_client.get_or_create_collection(name=collection_name)
        self.embedding_fn = embedding_fn

    def add_documents(self, documents):  # 文档向量化后入库（离线阶段）
        '''向 collection 中添加文档与向量'''
        # Chroma 会自动处理向量化，只要传入 embedding_fn
        self.collection.add(
            embeddings=self.embedding_fn(documents),  # 每个文档的向量
            documents=documents,  # 文档的原文
            ids=[f"id{i}" for i in range(len(documents))]  # 每个文档的唯一ID
        )

    def search(self, query, top_n):
        '''检索向量数据库'''
        results = self.collection.query(
            query_embeddings=self.embedding_fn([query]), # 对 Query 进行向量化
            n_results=top_n # 返回最相似的 Top-N 个结果
        )
        return results


**测试检索：**



<div class="alert alert-success"> <b>澄清几个关键概念：</b><ul> <li>向量数据库的意义是快速的检索；</li> <li>向量数据库本身不生成向量，向量是由 Embedding 模型产生的；</li> <li>向量数据库与传统的关系型数据库是互补的，不是替代关系，在实际应用中根据实际需求经常同时使用。</li> </ul> </div>

> **【🔍 深度解析 & 关联性解读】**
> 
> - `embedding_fn` 的传递：注意 `MyVectorDBConnector` 构造函数中传入了 `embedding_fn`。在 `add_documents` 时，Chroma 内部调用它将文档转向量；在 `search` 时，Chroma 再次调用它将 `query` 转向量。**这确保了文档和查询使用同一个模型，处于同一个向量空间，这是检索成功的根本前提。**
>     
> - **持久化**：代码注释中提到了 `PersistentClient`。内存模式仅供学习，企业实战必须使用持久化模式或 Server 模式，否则重启后知识库就清空了。
>     

#### 3.3.2 Chroma 的服务模式

- **Server/Client 模式**：类似于 MySQL，启动一个 Chroma 服务端，代码通过 HTTP 连接。适合生产环境分布式部署，便于扩容和集群管理。 Python
    

Python

In [15]:
# 创建数据库连接
vector_db = MyVectorDBConnector("demo", get_embeddings)
# 离线步骤：灌库（将文档转为向量并存储）
vector_db.add_documents(paragraphs)  # 文档向量化后入库（离线阶段）

# 在线步骤：检索
user_query = "Llama 2有多少参数" 
# Does Llama 2 have a conversational variant
results = vector_db.search(user_query, 2) # 找最相似的2个片段

for para in results['documents'][0]:
    print(para+"\n")

/3T/strange007/miniconda3/envs/teach/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


 1. Llama 2, an updated version of Llama 1, trained on a new mix of publicly available data. We also increased the size of the pretraining corpus by 40%, doubled the context length of the model, and adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants of Llama 2 with 7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper but are not releasing.§

 In this work, we develop and release Llama 2, a family of pretrained and ﬁne-tuned LLMs, Llama 2 and Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested, Llama 2-Chat models generally perform better than existing open-source models. They also appear to be on par with some of the closed-source models, at least on the human evaluations we performed (see Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-speciﬁc data annotation and tuning, as well as conducting red-teaming and emp

In [ ]:
# !chroma run --path ./db_path --port 8001
# 这个命令用于启动 ChromaDB 服务，但是需要在终端中运行，而不是在 Jupyter Notebook 中运行。
# 否则会让jupyter Notebook 卡死，无法继续执行后续代码。


Python

In [18]:
import chromadb  # 向量数据库（Chromadb）
chroma_client = chromadb.HttpClient(host='localhost', port=8001)

ValueError: Could not connect to a Chroma server. Are you sure it is running?


#### 3.3.3 主流向量数据库功能对比

### 向量数据库功能对比表

<table><tbody><tr><td><p><strong>Name</strong></p></td><td><p><strong>Web GUI</strong></p></td><td><p><strong>GPU Support</strong></p></td><td><p><strong>Remote Support (HTTP/gRPC)</strong></p></td><td><p><strong>Cloud Native</strong></p></td><td><p><strong>Opensource</strong></p></td><td><p><strong>Metadata (hybrid search)</strong></p></td></tr><tr><td><p>FAISS</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>N</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>N</p></td></tr><tr><td><p>Pinecone</p></td><td><p>Y</p></td><td><p>N/A</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>N</p></td><td><p>Y</p></td></tr><tr><td><p>Milvus</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td></tr><tr><td><p>Weaviate</p></td><td><p>N</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td></tr><tr><td><p>Qdrant</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td></tr><tr><td><p>PGVector</p></td><td><p>N</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td></tr><tr><td><p>RediSearch</p></td><td><p>N</p></td><td><p>N</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td><td><p>Y</p></td></tr><tr><td><p><strong>Chroma</strong></p></td><td><p><strong>N</strong></p></td><td><p><strong>N</strong></p></td><td><p><strong>Y</strong></p></td><td><p><strong>Y</strong></p></td><td><p><strong>Y</strong></p></td><td><p><strong>Y</strong></p></td></tr></tbody></table>


**选型建议：**

- **Milvus**：功能强大，支持分布式，适合海量数据（亿级），推荐企业级生产使用。
    
- **Chroma**：轻量，Python 友好，适合快速原型开发和中小规模应用。
    
- **ElasticSearch**：老牌搜索霸主，现在也支持向量检索（Vector Search）。如果企业已有 ES 设施，可以直接复用，且支持混合检索。
    

### 3.4 动手实现一个 RAG Bot

现在我们将前面的所有模块串联起来，实现一个最基础的 RAG 机器人类。



> **【🔍 深度解析 & 关联性解读】**
> 
> - **代码闭环**：`RAG_Bot` 类是本课程前三章内容的集大成者。
>     
>     - `self.vector_db.search` 对应了 **Retrieval（检索）**。
>         
>     - `build_prompt` 对应了 **Augmented（增强）**。
>         
>     - `self.llm_api(prompt)` 对应了 **Generation（生成）**。
>         
> - **Top-K 的选择**：这里默认 `n_results=2`。在实际业务中，这个值不能太小（信息不全），也不能太大（超过 LLM 窗口限制或引入噪音）。这是一个需要根据切分粒度进行调优的超参数。
>     


In [12]:
class RAG_Bot:
    def __init__(self, vector_db, llm_api, n_results=2):
        self.vector_db = vector_db
        self.llm_api = llm_api
        self.n_results = n_results

    def chat(self, user_query):
        # 1. 检索 (Retrieval)
        # Top-K 的选择：这里获取self.n_results条最相似的文档片段
        search_results = self.vector_db.search(user_query, self.n_results)  # 向量检索 Top-K（在线阶段）

        # 2. 增强 (Augmented): 构建 Prompt
        # search_results['documents'][0] 是一个包含多个文本片段的列表
        prompt = build_prompt(  # 将检索结果与用户问题填入 Prompt 模板
            prompt_template, 
            context=search_results['documents'][0], 
            query=user_query
        )

        # 3. 生成 (Generation): 调用 LLM
        response = self.llm_api(prompt)
        return response

# 实例化 RAG 机器人
bot = RAG_Bot(
    vector_db,
    llm_api=get_completion
)

user_query = "llama 2有多少参数?"
response = bot.chat(user_query)

print("==== RAG 回复 ====")
print(response)

{'context': [' 1. Llama 2, an updated version of Llama 1, trained on a new mix of publicly available data. We also increased the size of the pretraining corpus by 40%, doubled the context length of the model, and adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants of Llama 2 with 7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper but are not releasing.§', ' In this work, we develop and release Llama 2, a collection of pretrained and ﬁne-tuned large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters. Our ﬁne-tuned LLMs, called Llama 2-Chat, are optimized for dialogue use cases. Our models outperform open-source chat models on most benchmarks we tested, and based onour human evaluations for helpfulness and safety, may be a suitable substitute for closed source models. We provide a detailed description of our approach to ﬁne-tuning and safety improvements of Llama 2-Chat in order to en


### 3.5 OpenAI 的 Embedding 模型 (进阶)

2024年初，OpenAI 发布了 `text-embedding-3-large` 和 `small`。 其最大特点是支持 **Matryoshka Representation Learning**，即支持**自定义缩短向量维度**。

- **越大越准**：保留全部维度（如 3072），特征提取更丰富。
    
- **越小越快**：截断维度（如 256），虽然精度略有下降，但存储和检索速度大幅提升。
    

<img src="./assets/mteb.png" style="margin-left: 0px" width=600px>



<div class="alert alert-info"> <b>扩展阅读：这种可变长度的 Embedding 技术背后的原理叫做 Matryoshka Representation Learning</a> </b> </div>

https://arxiv.org/abs/2205.13147


In [13]:
model = "text-embedding-3-large"
dimensions = 256 # 我们可以强制指定一个较小的维度

query = "国际争端"
documents = [
    "联合国就苏丹达尔富尔地区大规模暴力事件发出警告",
    "土耳其、芬兰、瑞典与北约代表将继续就瑞典“入约”问题进行谈判",
    "日本岐阜市陆上自卫队射击场内发生枪击事件 3人受伤",
    "国家游泳中心（水立方）：恢复游泳、嬉水乐园等水上项目运营",
    "我国首次在空间站开展舱外辐射生物学暴露实验",
]

query_vec = get_embeddings([query], model=model, dimensions=dimensions)[0]  # 将文本转为向量
doc_vecs = get_embeddings(documents, model=model, dimensions=dimensions)  # 将文本转为向量

# 验证维度是否变成了 256
print("向量维度: {}".format(len(query_vec))) 

print("Query与Documents的余弦距离:")
for vec in doc_vecs:
    print(cos_sim(query_vec, vec))

向量维度: 256
Query与Documents的余弦距离:
0.27710082224825
0.3358965491857587
0.12354792500287116
0.166938966813982
0.12346200657769162



* * *

## 四、实战 RAG 系统的进阶知识

在实际生产环境中，简单的“按换行符切分+向量检索”往往效果不佳。以下是**面试和实战中的重灾区**。

### 4.1 文本分割的粒度 (Chunking Strategy)

**缺陷**： 如果简单按换行符切分，可能会出现：

1. **粒度太大**：包含了大量无关信息，干扰 LLM。
    
2. **粒度太小**：信息支离破碎，丢失语义。
    
3. **跨片段问题**：问题的答案前半句在片段 A，后半句在片段 B，导致检索时只能召回其中一半，上下文不完整。
    

**演示问题：** 我们之前的代码是按自然段落切分的。如果我们问“Llama 2 Chat 有多少参数”，检索结果可能只包含笼统的介绍，而不包含具体的参数列表。

**改进方案：重叠切分 (Overlap Chunking)** 我们按照固定的字符数（Chunk Size）切分，并让相邻的片段之间有一定的重叠（Overlap）。这样可以保证语义的连续性。

**设置建议**：

- **Chunk Size**：一般设置为 300-2000，取决于你的 LLM 上下文窗口。
    
- **Overlap Size**：一般设置为 Chunk Size 的 10%-20%。
    

### 4.1 文本切分（Chunking）：为什么不能整篇入库？

**问题**：整篇文档直接向量化，一个向量承载的信息过载；检索时召回整篇文档，LLM 的上下文窗口也装不下。RAG 的检索最小单元应该是"**文本块（Chunk）**"。

- **块太小**：丢失上下文语义，检索不准
- **块太大**：检索不精准，且塞给 LLM 会浪费 Token
- 经验值：`chunk_size` 约 300 字符，`overlap_size` 取 chunk_size 的 10%-20%

**为什么要重叠（Overlap）？**
一句话可能横跨两个块的边界，被切断后语义残缺。让相邻块重叠一部分，边界处的上下文就不会丢失。

**切分单位**：用 NLTK 的 `sent_tokenize` 按"句子"切（而非简单按字符/句号切），保证块边界落在完整句子上。


In [ ]:
import nltk  # NLTK 自然语言处理库（用于分句）

# 下载分句所需数据包（sent_tokenize 依赖的预训练模型）
nltk.download('punkt_tab')  # NLTK 3.8.2+ 必需的新数据包
nltk.download('punkt')      # 经典分句模型（保险起见一并下载）

# 若下载超时/失败（国内网络），可先设置代理再重试：
# import os
# os.environ["http_proxy"] = "http://127.0.0.1:1081"
# os.environ["https_proxy"] = "http://127.0.0.1:1081"
# nltk.download('punkt_tab'); nltk.download('punkt')


#### `sent_tokenize` 是什么？

- **含义**：Sentence Tokenize（句子分词/分句）。
    
- **作用**：它是一个智能工具，用于将一段长文本（段落）**按句子**进行拆分。
    
- **为什么不用** `text.split('.')`？：简单的按句号拆分会出错。例如 *"Mr. Smith is happy."*，如果按句号拆分，*"Mr."* 会被误认为是一句话的结束。`sent_tokenize` 使用训练好的模型来识别缩写、标题等，能正确判断句子的边界。
    

#### `nltk.download('punkt')` 和 `nltk.download('punkt_tab')` 是什么？

- `sent_tokenize` 函数不是纯粹的规则代码，它依赖于预先训练好的数学模型来判断句子在哪里结束。
    
- `punkt`：这是 NLTK 中经典的句子分词模型数据包。
    
- `punkt_tab`：这是较新版本的 NLTK（特别是 3.8.2 及以后）所依赖的文件结构包。如果不下载它，在调用分句功能时可能会报错（如 `Resource punkt_tab not found`）。
    
- **总结**：这两行代码是为了下载“大脑”，有了这两个包，`sent_tokenize` 才知道如何正确地断句。

### 4.1.1 `split_text` 代码结构分析（三步走）

```
句子: [s1] [s2] [s3] [s4] [s5] [s6] [s7] ...
块1:  [s1  s2  s3]                        ← 向前无重叠，向后填满 chunk_size
块2:  [s2  s3  s4  s5]                   ← 开头带上块1尾部(s2 s3) 作 overlap
块3:  [s4  s5  s6  s7]                   ← 重叠保证边界句子不丢失
```

1. **句子扁平化**：双重循环把"段落列表 → 句子列表"（`sent_tokenize` + 去首尾空格）
2. **向前回溯（Overlap）**：当前块开头向前拼接前几句，凑够 `overlap_size`，补上文
3. **向后填充（Chunk）**：不断追加后续句子，直到总长度达到 `chunk_size`

运行后得到的 `chunks` 列表，就是送入 Embedding 与向量库的检索单元。


In [ ]:
# chunk_size: 每个切分块的目标字符长度（比如一段话大概 300 字）
# overlap_size: 重叠长度。为了防止一句话被切断导致上下文丢失，
#               我们在块的开头会包含上一块结尾的一部分内容。
def split_text(paragraphs, chunk_size=300, overlap_size=100):
    '''按指定 chunk_size 和 overlap_size 交叠切割文本'''
    
    # 步骤 1: 数据清洗与扁平化
    # 输入的 paragraphs 是一个段落列表。
    # 这里做了一个双重循环：
    #   1. 遍历每个段落 p
    #   2. 用 sent_tokenize(p) 把段落拆成句子
    #   3. s.strip() 去除句子首尾空格
    # 最终 sentences 变成了一个包含所有句子的长列表
    sentences = [s.strip() for p in paragraphs for s in sent_tokenize(p)]  # 按句子切分文本
    
    chunks = [] # 用于存放最终切分好的文本块
    i = 0       # 当前处理到的句子索引
    
    # 步骤 2: 循环处理每个句子，组装 Chunk
    while i < len(sentences):
        # 当前块的核心起始句子
        chunk = sentences[i]
        
        # --- 向前回溯（处理 Overlap 重叠部分）---
        # 这一步是为了让当前块不仅仅从第 i 句开始，还包含 i 之前的几句话，
        # 直到凑够 overlap_size 长度。这样当前块的开头就有上文信息了。
        overlap = ''
        prev = i - 1
        # 如果还有前一句 (prev >= 0) 且 加上前一句后的长度没超过设定的重叠限制
        while prev >= 0 and len(sentences[prev]) + len(overlap) <= overlap_size:
            # 把前一句拼接到 overlap 的前面
            overlap = sentences[prev] + ' ' + overlap
            prev -= 1 # 继续往前找
        
        # 将重叠部分拼接到当前核心句子的前面
        chunk = overlap + chunk
        
        # --- 向后延伸（填充 Chunk 直到达到 chunk_size）---
        next = i + 1
        # 如果还有下一句 且 加上下一句后总长度没超过 chunk_size
        while next < len(sentences) and len(sentences[next]) + len(chunk) <= chunk_size:
            # 把下一句拼接到当前块的后面
            chunk = chunk + ' ' + sentences[next]
            next += 1 # 继续往后找
        
        # 将组装好的这就话加入结果列表
        chunks.append(chunk)
        
        # --- 移动指针 ---
        # 关键点：下一次循环从哪里开始？
        # 从 next 开始。因为 [i ... next-1] 这些句子已经被包含在当前的主体内容里了。
        # 下一个 chunk 会从 next 开始作为核心，然后它会再次“回头”看（执行上面的 overlap 逻辑）
        # 把刚才当前块结尾的部分句子作为它的开头重叠部分。
        i = next
        
    return chunks


<div class="alert alert-info"> <b>进阶提示：</b> 对于非常复杂的文本（如医疗文献、法律文书），简单的规则切割可能仍然不够。此时可以使用 <b>NSP (Next Sentence Prediction)</b> 任务微调模型，来智能判断两个段落是否应该合并。 </div>

**应用改进后的切分：**



你会发现，通过合理的切分，检索到的内容更精准，模型的回答也更完善（例如能准确列出 7B, 13B, 70B 三种变体）。

> **【🔍 深度解析 & 关联性解读】**
> 
> - **重叠切分的意义**：在 `split_text` 函数中，`overlap_size` 确保了如果一个句子被切断，它的一部分会出现在前一个块，另一部分出现在后一个块。这样无论检索到哪个块，LLM 都能获得上下文。
>     
> - **对比效果**：使用 2.1 节的简单切分时，LLM 只能回答笼统的“最高 70B”；使用这里的重叠切分后，LLM 成功列出了“7B, 13B, 70B”三个具体参数。这证明了**切分策略直接影响 RAG 的回答质量**。
>     


In [15]:
# 使用新的切分策略
chunks = split_text(paragraphs, 300, 100)

# 重建数据库并灌入新的 chunks
vector_db = MyVectorDBConnector("demo_text_split", get_embeddings)
vector_db.add_documents(chunks)  # 文档向量化后入库（离线阶段）

# 重建 Bot
bot = RAG_Bot(vector_db, get_completion)

user_query="llama 2 chat有多少参数"
# 检索 Top-2
search_results = vector_db.search(user_query, 2)  # 向量检索 Top-K（在线阶段）
for doc in search_results['documents'][0]:
    print(doc+"\n")

response = bot.chat(user_query)
print("==== 改进后的回复 ====")
print(response)

2. Llama 2-Chat, a ﬁne-tuned version of Llama 2 that is optimized for dialogue use cases. We release variants of this model with 7B, 13B, and 70B parameters as well. We believe that the open release of LLMs, when done safely, will be a net beneﬁt to society.

In this work, we develop and release Llama 2, a family of pretrained and ﬁne-tuned LLMs, Llama 2 and Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested, Llama 2-Chat models generally perform better than existing open-source models.

==== 改进后的回复 ====
Llama 2-Chat 有 7B、13B 和 70B 参数的变体。



### 4.2 检索后排序 (Rerank)

**问题**：向量检索（Top-K）是基于语义相似度的，但“语义相似”不等于“最匹配答案”。 有时候，最相关的文档在向量空间中可能排在第 3 或第 5 位，而不是第 1 位。而 LLM 对 Prompt 开头和结尾的内容更敏感，如果把相关文档排在中间，可能被忽略。

**方案**：**两阶段检索 (Retrieval + Rerank)**。

1. **粗排 (Retrieval)**：先用向量检索快速召回 Top-N（比如前 50 个）文档。这一步速度快，目的是**召回率**。
    
2. **精排 (Rerank)**：使用更精准的 **Cross-Encoder** 模型（排序模型）对这 50 个文档与 Query 进行逐一打分，重新排序，取分数最高的 Top-K（比如前 3 个）给 LLM。这一步计算慢，但**准确率**高。
    

<img src="./assets/sbert-rerank.png" style="margin-left: 0px" width=600px>

<div class="alert alert-danger"> 注意：以下代码需要下载模型，请确保网络通畅（能访问 Hugging Face）。 </div>


### 4.2.1 Rerank 代码原理：先召回、再精排

向量检索（BiEncoder）速度快但精度有限，只适合"**粗召回**"；CrossEncoder 精度高但速度慢，适合对少量候选做"**精排**"。

流程分三步：
1. **召回**：先用向量检索召回多一点候选（Top-5，宁多勿漏）
2. **打分**：把 `(query, doc)` 拼成一对交给 CrossEncoder，一次性输出相关度分数
3. **排序**：按分数从高到低重新排列，Top-K 结果更贴近真实意图

模型选择：英文 `ms-marco-MiniLM-L6-v2`（轻量快）、中文 `BAAI/bge-reranker-large`（效果好较大）。



**结论**：Rerank 是提升 RAG 准确率最有效的手段之一，是企业级 RAG 的标配。

> **【🔍 深度解析 & 关联性解读】**
> 
> - **Bi-Encoder vs Cross-Encoder**：
>     
>     - 前文的 **向量检索** 使用的是 Bi-Encoder（分别计算 Query 和 Doc 的向量，速度快）。
>         
>     - 这里的 **Rerank** 使用的是 Cross-Encoder（将 Query 和 Doc 拼在一起输入模型，计算注意力，更精准但速度慢）。
>         
> - **漏斗模型**：这是一个经典的漏斗筛选过程。第一层（向量检索）是大漏斗，快速从百万文档选出 50 个；第二层（Rerank）是精细筛子，从 50 个里选出最棒的 3 个给 LLM。
>     



In [ ]:
# 安装：pip install sentence_transformers
from sentence_transformers import CrossEncoder

# 加载重排序模型
# 英文推荐：cross-encoder/ms-marco-MiniLM-L-6-v2（轻量、速度快）
# 中文推荐：BAAI/bge-reranker-large（效果好但模型较大）
model_path = "/3T/strange007/models/ms-marco-MiniLM-L6-v2"
model = CrossEncoder(model_path, max_length=512)  # 加载重排序（Rerank）模型
print("✅ 重排序模型加载完成")


In [ ]:
user_query = "how safe is llama 2"

# ① 向量检索先召回多一点（Top-5，宁多勿漏）
search_results = vector_db.search(user_query, 5)
documents = search_results['documents']
print(f"召回 {len(documents)} 个候选片段")

# ② CrossEncoder 逐对打分：输入 [(query, doc1), (query, doc2), ...]
scores = model.predict([(user_query, doc) for doc in documents])  # 计算 query 与每个文档片段的相关度得分

# ③ 按分数从高到低排序（精排）
sorted_list = sorted(zip(scores, documents), key=lambda x: x, reverse=True)

print("==== Rerank 后的顺序 ====")
for score, doc in sorted_list:
    print(f"{score:.4f}\t{doc[:60]}...")


* * *

## 五、进阶难点：PDF 表格处理 (Multimodal RAG)

普通的 PDF 解析库（如 `pdfminer`）通常会把表格里的文字拆散，变成无意义的乱码，导致 RAG 无法回答有关表格数据的问题（例如：“在这个表格中，哪个模型的得分最高？”）。

<img src="./assets/table_rag.png" style="margin-left: 0px; background-color: white; padding: 10px;" width="800px">

**解决方案：多模态 RAG 流程**

1. **PDF 转图片**：将 PDF 页面转换为图片。
    
2. **表格检测**：使用视觉模型（如 Microsoft Table Transformer）识别图片中的表格区域。
    
3. **截图**：将表格区域裁剪出来，保存为单独的图片。
    
4. **多模态理解**：将表格图片发送给 **GPT-4o (Vision)**，让它用自然语言描述表格内容（例如：“这是一个模型性能对比表，Llama 2 得分为...”）。
    
5. **向量化存库**：将 GPT-4o 生成的“文本描述”向量化存入数据库，同时在元数据中保存原图片的路径。
    
6. **检索**：当用户提问时，检索到的是“文本描述”，但可以将原图片展示给用户，或再次把图片喂给 GPT-4o 进行精准回答。
    

### 5.1 步骤演示

**1\. PDF 转图片**


### 5.1 把 PDF 页面变成图片（PyMuPDF）

**为什么表格要先转成图片？**
纯文本解析（`pdfminer`）只能拿到文字流，**表格的版式结构会丢失**——行、列、单元格的关系全没了。多模态 RAG 的思路是：把 PDF 页面**渲染成图片**，让"看得见"的模型（Table Transformer / GPT-4o）去识别表格。

**PyMuPDF（导入名 `fitz`）**：PDF 解析与渲染库。核心操作：
- `fitz.open(pdf_file)`：打开 PDF，得到 Document 对象
- `page.get_pixmap()`：把一页"拍照"成位图（像素数据）
- PIL `Image.frombytes`：把位图转成图像对象，再 `save` 成 PNG

输出：每页一张 `page_N.png`，作为后续表格检测的输入。


### 5.1.1 `pdf2images` 代码结构分析

1. **建输出目录**：`os.path.splitext("llama2_page8.pdf")` 取文件名作为文件夹名
2. **打开 PDF**：`fitz.open(pdf_file)` 得到 Document 对象
3. **逐页渲染**：遍历 `page_count`，每页 `get_pixmap()` 生成位图 → PIL 转图像 → 保存 `page_N.png`
4. **收尾**：`pdf_document.close()` 释放资源

> 若页面文字偏小，可在 `get_pixmap()` 加 `matrix=fitz.Matrix(2, 2)` 放大 2 倍渲染。


In [ ]:
import fitz  # PyMuPDF 库，核心 PDF 处理库。注意：安装包叫 PyMuPDF，导入名叫 fitz
from PIL import Image # Pillow 库，用于处理图像数据的转换和保存
import os # Python 标准库，用于处理文件路径和目录创建

def pdf2images(pdf_file):
    '''
    函数功能：将指定 PDF 文件的每一页转换为 PNG 图片
    参数 pdf_file: PDF 文件的路径 (字符串)
    '''
    
    # 1. 准备输出目录
    # os.path.splitext("document.pdf") 会返回 ("document", ".pdf")
    # 这里只取第一个元素，即文件名（不含扩展名），作为输出文件夹的名字
    output_directory_path, _ = os.path.splitext(pdf_file)
    
    # 检查文件夹是否存在，如果不存在则创建
    if not os.path.exists(output_directory_path):
        os.makedirs(output_directory_path)
    
    # 2. 打开 PDF 文档
    # 使用 PyMuPDF 打开文件，生成一个 Document 对象
    pdf_document = fitz.open(pdf_file)  # 打开 PDF 文档
    
    # 3. 遍历每一页
    # pdf_document.page_count 获取总页数
    for page_number in range(pdf_document.page_count):
        # 获取当前页对象
        page = pdf_document[page_number]
        
        # 4. 将 PDF 页面渲染为像素图 (Rasterization)
        # get_pixmap() 是核心方法，它将矢量格式的 PDF 页面“拍照”变成位图数据
        # 默认分辨率是 72 DPI，如果觉得模糊，可以在这里添加参数 matrix=fitz.Matrix(2, 2) 来放大两倍
        pix = page.get_pixmap()
        
        # 5. 将 PyMuPDF 的像素数据转换为 PIL 图像对象
        # "RGB": 指定颜色模式
        # [pix.width, pix.height]: 指定图像尺寸
        # pix.samples: 实际的像素字节流数据
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        
        # 6. 保存图片到磁盘
        # 格式化路径：./文件夹名/page_1.png
        save_path = f"./{output_directory_path}/page_{page_number + 1}.png"
        image.save(save_path)
        print(f"已保存: {save_path}") # (可选) 打印进度
        
    # 7. 关闭文档，释放资源
    pdf_document.close()

# 调用函数示例
# 假设当前目录下有一个名为 llama2_page8.pdf 的文件
pdf2images("llama2_page8.pdf")

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

def show_images(dir_path):
    '''显示目录下的 PNG 图像'''
    for file in os.listdir(dir_path):
        if file.endswith('.png'):
            img = Image.open(os.path.join(dir_path, file))   # 读取页面图片
            plt.imshow(img)
            plt.axis('off')  # 不显示坐标轴
            plt.show()

# 显示刚渲染出的 PDF 页面（确认图片清晰、版式完整）
show_images("llama2_page8")


### 5.2 表格检测（Table Transformer）

**问题**：PDF 渲染成图片后，怎么知道"哪块区域是表格"？

**思路**：目标检测——把"表格"当作图中的目标框（bbox）找出来，用的是 DETR 架构的检测模型 `microsoft/table-transformer-detection`（类别只有 table / table rotation）。

**输入预处理**（模型要求）：
- `MaxResize(800)`：保持长宽比，限制最大边 800 像素（模型最优尺度）
- `ToTensor()`：PIL → 张量，像素值归一到 [0,1]
- `Normalize`：ImageNet 均方差归一化（模型在 ImageNet 预训练基础上微调，输入分布必须一致）

**输出后处理**：
- 模型输出相对坐标 `(cx, cy, w, h)` → 转换为像素坐标 `(x1, y1, x2, y2)`
- 解析出 label（类别）、score（置信度）、bbox（边界框），过滤 `no object`


### 5.2.1 表格检测代码结构（六个逻辑块）

| 块 | 内容 | 职责 |
|---|---|---|
| 1 | `MaxResize` 类 | 缩放工具：最大边限制 800px |
| 2 | `transforms.Compose` | 预处理管道：缩放 + 转张量 + 归一化 |
| 3 | 模型加载 | `AutoModelForObjectDetection.from_pretrained` |
| 4 | 坐标后处理 | `cxcywh→xyxy`、相对→像素、解析 label/score/bbox |
| 5 | 主函数 | 读图 → 预处理 → 推理 → 裁剪 → 保存 `table_images/` |
| 6 | 执行调用 | 检测 `page_1.png` 并保存裁剪结果 |

按上面的结构，代码拆成 6 个小块逐段运行，方便课堂演示与排错。



**2\. 识别并裁剪表格 (使用 Table Transformer)**

这一步使用了深度学习模型来“看”表格。这部分代码较复杂，涉及 PyTorch 的张量处理。


📁 **数据集：Llama 2 论文第 8 页表格图片集（多模态 RAG 示例）**

- 位置：`llama2_page8/table_images/` 目录下的 `.png` 图片文件
- 来源：由 PDF 表格检测模型从 `llama2.pdf` 第 8 页裁剪出的表格截图（每个表格一张图）
- 用途：演示多模态 RAG——先把表格图片用 GPT-4 Vision 转成文本描述，再向量化入库，实现"对表格提问"
- 处理方式：遍历目录收集图片路径 → 调用图像理解模型生成描述 → 存入向量数据库

In [ ]:
import torch
from PIL import Image
import os
import torchvision.transforms as transforms
from transformers import AutoModelForObjectDetection

# ========== 块1：图像预处理工具类 ==========

class MaxResize(object):
    '''
    自定义缩放类：保持长宽比，限制最大边不超过 max_size。
    Table Transformer 模型通常在 800-1000 像素左右的尺度下表现最好。
    '''
    def __init__(self, max_size=800):
        self.max_size = max_size

    def __call__(self, image):
        width, height = image.size
        current_max_size = max(width, height)   # 取长宽中的最大值
        scale = self.max_size / current_max_size  # 计算缩放比例
        resized_image = image.resize(            # 按比例缩放
            (int(round(scale * width)), int(round(scale * height)))
        )
        return resized_image


In [ ]:
# ========== 块2：预处理管道（Pipeline） ==========

detection_transform = transforms.Compose(
    [
        # 第一步：限制最大边 800px，防止图片过大导致显存溢出或推理变慢
        MaxResize(800),

        # 第二步：PIL 图像 (H,W,C) → 张量 (C,H,W)，像素值归一化到 [0,1]
        transforms.ToTensor(),

        # 第三步：ImageNet 均方差归一化（预训练模型的输入要求）
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)
print("✅ 预处理管道构建完成")


In [ ]:
# ========== 块3：加载检测模型 ==========

# 本地模型路径；若未下载，可改为 Hugging Face 模型 ID：
# model_path = "microsoft/table-transformer-detection"
model_path = "/3T/strange007/models/table-transformer-detection"

model = AutoModelForObjectDetection.from_pretrained(model_path)  # 加载目标检测模型
print("✅ 表格检测模型加载完成")


In [ ]:
# ========== 块4：坐标转换与后处理工具函数 ==========

def box_cxcywh_to_xyxy(x):
    '''
    坐标格式转换：
    模型输出 (cx, cy, w, h) → (中心点x, 中心点y, 宽度, 高度)
    转换为     (x1, y1, x2, y2) → (左上角x, 左上角y, 右下角x, 右下角y)
    方便后续用 PIL 的 crop 函数裁剪。
    '''
    x_c, y_c, w, h = x.unbind(-1)  # 拆分最后一维
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),  # 左上角
         (x_c + 0.5 * w), (y_c + 0.5 * h)]  # 右下角
    return torch.stack(b, dim=1)


def rescale_bboxes(out_bbox, size):
    '''
    坐标还原：模型输出是相对坐标（0~1 小数），
    乘以图片实际宽高，还原为像素坐标。
    '''
    width, height = size
    boxes = box_cxcywh_to_xyxy(out_bbox)   # 先转格式
    boxes = boxes * torch.tensor([width, height, width, height], dtype=torch.float32)
    return boxes


In [ ]:
def outputs_to_objects(outputs, img_size, id2label):
    '''
    解析模型输出：提取 Label（类别）、Score（置信度）、BBox（边界框）。
    outputs.logits 形状为 [batch, num_queries, num_classes]。
    '''
    # softmax 计算每个预测框属于各类别的概率，取最大值
    m = outputs.logits.softmax(-1).max(-1)

    pred_labels = list(m.indices.detach().cpu().numpy())  # 概率最大的类别 ID
    pred_scores = list(m.values.detach().cpu().numpy())   # 对应的置信度
    pred_bboxes = outputs["pred_boxes"].detach().cpu()    # 预测边界框

    # 边界框还原为像素坐标
    pred_bboxes = [elem.tolist() for elem in rescale_bboxes(pred_bboxes, img_size)]

    objects = []
    for label, score, bbox in zip(pred_labels, pred_scores, pred_bboxes):
        class_label = id2label[int(label)]       # 类别名称（如 table）
        if not class_label == "no object":        # 过滤背景类
            objects.append({
                "label": class_label,
                "score": float(score),
                "bbox": [float(elem) for elem in bbox],
            })
    return objects


In [ ]:
# ========== 块5：核心主函数：检测并裁剪表格 ==========

def detect_and_crop_save_table(file_path):
    # 1. 加载图片（确保 RGB 三通道）
    image = Image.open(file_path).convert("RGB")
    filename, _ = os.path.splitext(os.path.basename(file_path))  # 取文件名（无扩展名）

    # 2. 准备输出目录：图片同级下创建 table_images 文件夹
    cropped_table_directory = os.path.join(os.path.dirname(file_path), "table_images")
    if not os.path.exists(cropped_table_directory):
        os.makedirs(cropped_table_directory)

    # 3. 预处理：[C,H,W] → unsqueeze(0) 增加 batch 维 → [1,C,H,W]
    pixel_values = detection_transform(image).unsqueeze(0)

    # 4. 模型推理（no_grad 节省内存）
    with torch.no_grad():
        outputs = model(pixel_values)

    # 5. 后处理：解析检测结果
    id2label = model.config.id2label
    id2label[len(model.config.id2label)] = "no object"  # DETR 架构特性：末尾补背景类
    detected_tables = outputs_to_objects(outputs, image.size, id2label)
    print(f"检测到的表格数量: {len(detected_tables)}")

    # 6. 裁剪并保存每个表格
    for idx in range(len(detected_tables)):
        bbox = detected_tables[idx]["bbox"]       # [x1, y1, x2, y2]
        cropped_table = image.crop(bbox)          # PIL crop 裁剪
        save_path = os.path.join(cropped_table_directory, f"{filename}_{idx}.png")
        cropped_table.save(save_path)
        print(f"表格已保存至: {save_path}")


In [ ]:
# ========== 块6：执行调用 ==========
# 前提：已运行 pdf2images 把 llama2_page8.pdf 渲染成图片（llama2_page8/page_1.png）
detect_and_crop_save_table("llama2_page8/page_1.png")

# 显示裁剪出的表格图片
show_images("llama2_page8/table_images")


### 5.3 用 GPT-4o 看懂表格图片

表格被裁剪出来后，怎么让大模型"看懂"？

- 图片不能直接作为文本输入 → 先转 **base64 编码**，通过 `data:image/jpeg;base64,...` 随请求发送
- `client.chat.completions.create` 的 `content` 里同时放 `text`（问题）和 `image_url`（图片）
- `image_qa(query, image_path)`：把"问题 + 图片"交给 gpt-4o，返回文本回答

下面定义编码与问答函数，并直接对裁剪出的表格图片提问测试。


In [ ]:
import base64

def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

def image_qa(query, image_path):
    '''发送图片和问题给 GPT-4o'''
    base64_image = encode_image(image_path)
    response = client.chat.completions.create(  # 调用 LLM 接口生成回答
        model="gpt-4o",
        temperature=0,
        messages=[{
            "role": "user",
              "content": [
                  {"type": "text", "text": query},
                  {
                      "type": "image_url",
                      "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
                  },
              ],
        }],
    )
    return response.choices[0].message.content

# 测试直接对表格图片提问
response = image_qa("哪个模型在AGI Eval数据集上表现最好？得分多少？", "./llama2_page8/table_images/page_1_0.png")
print(response)

### 5.4 图像 → 文本描述 → 向量入库（多模态检索闭环）

向量数据库不能直接存图片，需要先"翻译"：

1. **图像描述**：用 `image_qa` 让 GPT-4o 把表格图片"描述成文本"
2. **文本向量化**：把描述文本交给 Embedding 模型转成向量
3. **入库**：向量 + 描述 + 原始图片路径（metadata）一起存入向量数据库
4. **检索**：用户文字提问 → 向量检索 → 命中描述 → 返回对应图片

`NewVectorDBConnector` 类正是对这一流程的封装（文本、图片两种入库方式）。



**5\. 向量化检索流程 (Summary)**

在真正的系统中，我们会调用 `image_qa` 函数，传入 Prompt `"请详细描述这张图片中的表格内容"`，得到一段文本摘要。然后将这段摘要向量化存入 ChromaDB。当用户问“AGI Eval 得分”时，就能检索到这段摘要，进而找到原图。

> **【🔍 深度解析 & 关联性解读】**
> 
> - **多模态 RAG 的核心难点**：传统的 Embedding 模型只能理解文本，无法“看懂”图片中的表格结构。
>     
> - **曲线救国策略**：我们通过“检测（Table Transformer） -> 截图 -> 视觉理解（GPT-4o） -> 生成文本描述 -> 向量化”这一长链路，将视觉信息转化回了我们熟悉的文本向量空间。
>     
> - **未来趋势**：虽然这个过程复杂，但目前是处理非结构化数据（表格、图表）最有效的方案之一。如果你不想手写这一套，推荐使用 **RAGFlow** 等开源框架，它们内部集成了类似的 DeepDoc 深度文档理解技术。
>     

### 推荐工具

如果你觉得手写这套表格处理流程太复杂，推荐使用现成的开源框架：

- **RAGFlow**：基于深度文档理解（DeepDoc）的开源 RAG 引擎，对表格、图表解析效果极佳，推荐在本地部署尝试。
    
- **Unstructured.io**：强大的非结构化数据解析库。
    
* * *

## 总结

### 课程核心回顾

我们通过两天时间，从零手写了一个 RAG 系统。请务必掌握以下流程，这是面试的重点：

**1\. 离线阶段 (Offline)**

- **加载**：解析 PDF/Word。
    
- **切分**：这是重点。使用 Chunk Size + Overlap，或者更高级的语义切分（NSP）。
    
- **向量化**：选择合适的 Embedding 模型（如 OpenAI 适合跨语言，BGE 适合中文）。
    
- **存储**：存入向量数据库（Chroma/Milvus）。
    

**2\. 在线阶段 (Online)**

- **查询转换**：用户问题向量化。
    
- **检索**：Top-K 相似度搜索。
    
- **重排序 (Rerank)**：**关键步骤**，使用 Cross-Encoder 提升精准度。
    
- **生成**：Prompt 组装 + LLM 回复。
    

In [ ]:
import chromadb  # 向量数据库（Chromadb）
from chromadb.config import Settings

class NewVectorDBConnector:
    '''
    封装 ChromaDB 的连接、初始化、数据添加和检索。
    统一处理文本与多模态（图像）数据的向量存储与检索。
    '''
    def __init__(self, collection_name, embedding_fn):
        '''
        collection_name: 集合名称
        embedding_fn: 外部传入的 Embedding 函数（文本 → 向量）
        '''
        # 初始化 ChromaDB 客户端（allow_reset 允许清空数据库，便于演示）
        chroma_client = chromadb.Client(Settings(allow_reset=True))
        chroma_client.reset()  # 清空旧数据（生产环境慎用）

        # 创建或获取集合（存储文档、元数据、向量的基本单位）
        self.collection = chroma_client.get_or_create_collection(name=collection_name)

        # 保存外部 Embedding 函数引用，供添加与查询时调用
        self.embedding_fn = embedding_fn
    def add_documents(self, documents):  # 文本向量化后入库（离线阶段）
        '''向集合中添加纯文本文档和对应的向量'''
        self.collection.add(
            embeddings=self.embedding_fn(documents),  # 文档 → 向量
            documents=documents,                      # 文档原文
            ids=[f"id{i}" for i in range(len(documents))],  # 唯一 ID
        )

    def add_images(self, image_paths):  # 图片向量化后入库（离线阶段）
        '''
        向集合中添加图像数据：
        先用 image_qa 把图片"翻译"成文本描述，再向量化入库。
        '''
        # 1. 多模态处理：每张图片 → GPT-4o 文本描述
        documents = [image_qa("请简要描述图片中的信息", image) for image in image_paths]

        self.collection.add(
            embeddings=self.embedding_fn(documents),      # 描述文本 → 向量
            documents=documents,                          # 描述文本
            ids=[f"id{i}" for i in range(len(documents))],
            metadatas=[{"image": image} for image in image_paths],  # 记录原图路径
        )
    def search(self, query, top_n):
        '''
        检索向量数据库，找到与查询最相似的 Top-N 结果。
        返回 dict：documents / embeddings / metadatas / distances 等。
        '''
        results = self.collection.query(
            query_embeddings=self.embedding_fn([query]),  # 查询文本 → 向量
            n_results=top_n                               # 返回数量
        )
        return results


In [ ]:
images = []
dir_path = "llama2_page8/table_images"
for file in os.listdir(dir_path):
    if file.endswith('.png'):
        # 打开图像
        images.append(os.path.join(dir_path, file))

new_db_connector = NewVectorDBConnector("table_demo",get_embeddings)
new_db_connector.add_images(images)  # 图片向量化后入库（离线阶段）

In [ ]:
query  = "哪个模型在AGI Eval数据集上表现最差。得分多少"

results = new_db_connector.search(query, 1)  # 向量检索 Top-K（在线阶段）
metadata = results["metadatas"][0]
print("====检索结果====")
print(metadata)
print("====回复====")
response = image_qa(query,metadata[0]["image"])
print(response)

---

# 六、进阶实战：企业级向量数据库 Milvus

前文我们用 **Chroma** 完成了 RAG 的最小闭环。这一章把检索底座升级为**企业级向量数据库 Milvus**，把《向量数据库原理与实战》课程的核心知识落地为可运行的代码：

- 为什么需要向量数据库（LLM 记忆 / 维度诅咒 / 传统库短板）
- Milvus 核心概念与 **Docker 一键部署**
- 建集合 → 向量入库 → 建索引 → 检索（以文搜图 / 以图搜图）
- 把 RAG 检索层从 Chroma 替换为 Milvus，形成完整闭环

> 素材来源：`VectorDatabase-Codes/向量数据库原理与实战-01.txt`（NoSQL 课程课件）


### 6.1 为什么需要向量数据库？

**痛点一：LLM 的"金鱼记忆"**
大模型的信息只存在于有限的上下文窗口里（GPT-3.5 约 3000 字、GPT-4 约 24000 字），长对话、长文档读到后面就忘掉前面——需要一个"外部大脑"永久记住关键信息。

**痛点二：高维数据的"维度诅咒"**
图片、语音、文本被 AI 转成高维向量后，维度越高数据越稀疏，特征被无关信息淹没，传统距离度量在极高维度下趋于失效。

**痛点三：传统数据库"水土不服"**
MySQL 靠 B+ 树做精确匹配，面对向量只能全表扫描。向量数据库用 **ANN 索引**（IVF / HNSW）把"全库暴力搜索"变成"先分桶、再局部搜索"，亿级数据也能毫秒级响应。


### 6.2 为什么选 Milvus？

| 产品 | 定位 | 特点 |
|---|---|---|
| FAISS | 开源工具库（Meta） | 检索极快、支持 GPU，但无分布式、无持久化管理 |
| **Milvus** | **云原生企业级向量数据库** | 高可用、水平扩容、混合检索、全生命周期管理 |
| Chroma | 嵌入式教学库 | 轻量易用，适合原型（前文已用） |

Milvus 是生产级 RAG 的标准选择：支持**向量 + 标量混合过滤**、分布式集群、完善的权限与监控体系。本课用它把前面的 Demo 升级为"可上生产"的形态。


### 6.3 Milvus 核心概念（类比 SQL）

| Milvus | 类比 MySQL | 说明 |
|---|---|---|
| Collection 集合 | 表 | 存储一组向量及其标量字段 |
| Field 字段 | 列 | 如 id、文本、向量 embedding |
| Entity 实体 | 行 | 一条具体数据 |
| Index 索引 | 索引 | 加速 ANN 检索（IVF/HNSW） |
| Partition 分区 | 分区表 | 按业务切分，加速过滤 |

上手口诀：**建库(Collection) → 插数据(Entity) → 建索引(Index) → 检索(Search)**，与 SQL 的"建表 → 插入 → 索引 → 查询"一一对应。


### 6.4 Docker 部署 Milvus（Standalone 模式）

Milvus Standalone 由三个容器组成：

- **milvus**：向量检索引擎（核心服务，客户端端口 **19530**，健康检查 9091）
- **etcd**：元数据存储（记录 Collection / Schema 信息）
- **minio**：对象存储（保存向量数据文件）

官方提供现成的 docker-compose 文件，一条命令即可启动整个集群。


In [ ]:
# ① 下载官方 compose 文件（Windows PowerShell / Linux / Mac 均可）
# curl -LO https://github.com/milvus-io/milvus/releases/download/v2.4.9/milvus-standalone-docker-compose.yml -O docker-compose.yml

# ② 后台启动三个容器（首次自动拉取镜像，约需几分钟）
# docker compose up -d

# ③ 查看状态：三个容器都应显示 running / healthy
# docker compose ps

# ④ 常用运维命令（本 cell 仅演示，取消注释后执行）
# docker compose stop          # 停止（保留数据）
# docker compose restart       # 重启
# docker compose down -v       # 彻底删除（含数据卷，慎用！）


### 6.5 安装客户端并连接

Python 端使用官方 SDK **pymilvus**。`open_clip_torch` 用于后面的图片向量化（只跑文本 RAG 可不装）。

连接成功后先检查服务版本，确认 Docker 容器已就绪。


In [ ]:
# 安装 Milvus 客户端 SDK 与 CLIP 图片向量化库
# pip install pymilvus open_clip_torch


In [ ]:
from pymilvus import connections, utility

# 连接本地 Docker 启动的 Milvus 服务（默认端口 19530）
connections.connect(host="127.0.0.1", port="19530")

print("Milvus 服务版本:", utility.get_server_version())
print("当前已存在的集合:", utility.list_collections())


### 6.6 实战一：图片向量检索（以文搜图 / 以图搜图）

素材沿用《向量数据库原理与实战-01》项目的 `local_images` 文件夹：在当前目录新建 `local_images`，放入若干图片（jpg/jpeg/png/bmp 均可）。

> 如果暂时没有图片：本节的代码会自动切换为**随机向量演示模式**，同样能跑通 Milvus "建库 → 入库 → 检索" 的完整流程，只是结果不具真实语义。


In [ ]:
import os

IMAGE_DIR = "local_images"
os.makedirs(IMAGE_DIR, exist_ok=True)  # 没有目录则创建

# 遍历文件夹收集图片
img_paths = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR)
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]

if img_paths:
    print(f"✅ 检测到 {len(img_paths)} 张图片")
else:
    print("⚠️ local_images 为空 → 将使用随机向量演示（放入图片后重跑后续 cell 即可）")


### 6.7 把图片变成向量：CLIP 双塔模型

**嵌入（Embedding）** 是把"万物翻译成数学语言"的核心一步：CLIP 同时把**图片**和**文本**映射到同一个 512 维向量空间，所以"蓝天白云"的文本向量会靠近蓝天的图片向量——这就是**以文搜图**的原理。

- 相比 Word2Vec 的**静态嵌入**（一词一向量、不分语境），CLIP/BERT 属于**动态嵌入**，根据上下文动态生成向量
- 向量做 **L2 归一化**后，内积等价于余弦相似度，检索统一用 COSINE 度量


In [ ]:
# 仅在有真实图片时才加载 CLIP（首次运行自动下载权重，需联网）
if img_paths:
    import numpy as np
    import open_clip
    import torch
    from PIL import Image

    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    model.eval()

    def img_embed(path):
        # 图片 → 512 维归一化向量
        img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0)
        with torch.no_grad():
            v = model.encode_image(img).numpy().astype("float32")
        v /= np.linalg.norm(v, axis=1, keepdims=True)  # L2 归一化（内积=余弦）
        return v

    def text_embed(text):
        # 文本 → 512 维归一化向量
        with torch.no_grad():
            v = model.encode_text(open_clip.tokenize([text])).numpy().astype("float32")
        v /= np.linalg.norm(v, axis=1, keepdims=True)
        return v

    print("✅ CLIP 模型加载完成")


### 6.8 建集合（Collection）

设计图片库字段结构（对应 SQL 的 `CREATE TABLE`）：

- `id`：主键，INT64，**auto_id 自动生成**
- `image_path`：图片路径（VARCHAR，长度 512）
- `embedding`：512 维向量（FLOAT_VECTOR，CLIP 输出维度）

> 可重复运行：若集合已存在，先 `drop_collection` 删除再重建。


In [ ]:
from pymilvus import Collection, CollectionSchema, FieldSchema, DataType, utility

# 可重复运行：集合已存在则先删除
if utility.has_collection("image_milvus"):
    utility.drop_collection("image_milvus")

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="image_path", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=512),
]
schema = CollectionSchema(fields, description="本地图片库 Milvus 向量检索")

col = Collection(name="image_milvus", schema=schema)
print("✅ 集合创建成功:", col.name)


### 6.9 向量入库（Insert）

把每条数据包装成 `{字段: 值}` 字典插入集合；`flush()` 将内存数据**落盘持久化**——对应 SQL 的 `INSERT` + 提交事务。`num_entities` 返回当前实体总数。


In [ ]:
import numpy as np  # 保证随机向量演示也可用

rows = []
if img_paths:
    # 真实图片：逐张提取 CLIP 向量
    for p in img_paths:
        rows.append({"image_path": p, "embedding": img_embed(p)[0].tolist()})
else:
    # 随机向量演示：生成 5 条 512 维归一化向量（仅演示 API 流程）
    rng = np.random.default_rng(42)
    for i in range(5):
        v = rng.standard_normal(512).astype("float32")
        v /= np.linalg.norm(v)
        rows.append({"image_path": f"demo_image_{i}.jpg", "embedding": v.tolist()})
    print("（演示模式）插入 5 条随机向量")

col.insert(rows)  # 插入实体（id 自动生成）
col.flush()       # 落盘持久化
print(f"✅ 已入库 {col.num_entities} 条向量")


### 6.10 建索引（IVF_FLAT）与 Load

**为什么需要索引？** 全库暴力搜索在海量数据下不可行。IVF（倒排文件）先对向量做 **K-Means 聚类**，分成 `nlist` 个桶；检索时只搜索离查询最近的 `nprobe` 个桶——"先定位书架，再细找书"。

- `nlist`：桶的数量（分得多细）
- `nprobe`：查询时打开几个桶（越大召回越高、越慢）

> HNSW（分层小世界图）是更优的图索引，生产环境常用；这里先用 IVF_FLAT 理解分桶思想。教学数据小，`nlist=16` 即可，生产通常设 1024+。


In [ ]:
# 建索引：IVF_FLAT 分桶索引 + COSINE 余弦度量
col.create_index("embedding", {
    "index_type": "IVF_FLAT",   # 倒排分桶索引
    "metric_type": "COSINE",    # 余弦相似度（归一化后等价内积）
    "params": {"nlist": 16},    # 聚类桶数
})
print("✅ 索引创建完成")

col.load()  # 将索引与数据载入内存，进入可查询状态
print("✅ 集合已 load，可开始检索")


### 6.11 以文搜图（Text → Image）

流程：查询文本 → CLIP 文本向量 → Milvus 检索 Top-K → 返回相似度得分 + 图片路径。

> `nprobe` 是查询参数：搜索时打开多少个桶。越大候选越多、召回越全，但耗时越长——这就是"速度 vs 召回率"的权衡。


In [ ]:
# 生成查询向量：有图片用 CLIP 文本向量，否则用随机向量
if img_paths:
    query, qv = "蓝天白云", text_embed("蓝天白云")
else:
    query = "demo"
    qv = np.random.default_rng(7).standard_normal(512).astype("float32")
    qv /= np.linalg.norm(qv)

# Milvus 检索 Top-3，返回相似度得分与图片路径
results = col.search(
    data=qv,                      # 查询向量
    anns_field="embedding",       # 检索字段
    param={"metric_type": "COSINE", "params": {"nprobe": 8}},  # nprobe=8 个桶
    limit=3,                      # Top-K
    output_fields=["image_path"]  # 回传标量字段
)

print(f"🔍 查询：{query}")
for hit in results[0]:
    print(f"  相似度 {hit.score:.4f} | {hit.entity.get('image_path')}")


### 6.12 以图搜图（Image → Image）

与文搜图**唯一区别**：查询向量来自参考图片（而非文本）。上传一张图，系统自动找到视觉上最相似的图片——复刻百度识图的核心逻辑。借助 Jupyter 内嵌显示，可以直接"看"到检索结果。


In [ ]:
if img_paths:
    from IPython.display import display, Image as IPImage

    ref = img_paths[0]  # 取第一张图片作为参考图
    res = col.search(
        data=img_embed(ref), anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 8}},
        limit=3, output_fields=["image_path"]
    )

    print("🔍 参考图片：", ref)
    display(IPImage(ref))  # 显示参考图
    for hit in res[0]:
        path = hit.entity.get("image_path")
        print(f"  相似度 {hit.score:.4f} | {path}")
        display(IPImage(path))  # 显示检索到的相似图片
else:
    print("（演示模式）无真实图片，跳过以图搜图；放入图片后重跑本节即可")


### 6.13 实战二：把 RAG 检索层从 Chroma 换成 Milvus

回到课程主线：前文用 Chroma 存储 `llama2.pdf` 切分出的文本块，这里改用 **Milvus 承载知识库**，切分 / Embedding / Prompt / LLM 全部复用前文函数：

`split_text(paragraphs, 300, 100)` → 文本块
`get_embeddings(chunks)` → 1536 维向量（OpenAI text-embedding-ada-002）
`build_prompt / prompt_template / get_completion` → 检索增强问答

> 依赖前文 cell 已运行（kernel 中保留 `paragraphs`、`split_text`、`get_embeddings`、`build_prompt` 等变量）。


In [ ]:
# 复用前文：对论文段落做 300 字切分（重叠 100 字，保证上下文连续）
try:
    chunks = split_text(paragraphs, 300, 100)
except NameError:
    raise SystemExit("请先运行前面章节的 cell（需要 paragraphs 与 split_text）")

print(f"共切出 {len(chunks)} 个文本块")
for i, c in enumerate(chunks[:2]):  # 抽查前 2 块，确认数据就绪
    print(f"--- 块{i+1} ---")
    print(c[:120])


In [ ]:
# 可重复运行：删除旧集合
if utility.has_collection("rag_llama2_milvus"):
    utility.drop_collection("rag_llama2_milvus")

# 知识库集合：id + 文本块 + 1536 维向量（OpenAI ada-002 输出维度）
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="chunk_text", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=1536),
]
rag_col = Collection("rag_llama2_milvus", CollectionSchema(fields, "Llama2 论文知识库"))

# ① 文本块 → OpenAI Embedding 向量
embeddings = get_embeddings(chunks)

# ② 入库（id 自动生成）+ 落盘
rows = [{"chunk_text": t, "embedding": e} for t, e in zip(chunks, embeddings)]
rag_col.insert(rows)
rag_col.flush()

# ③ 建索引 + load（进入可查询状态）
rag_col.create_index("embedding", {
    "index_type": "IVF_FLAT", "metric_type": "COSINE", "params": {"nlist": 16}})
rag_col.load()

print(f"✅ 知识库就绪：{rag_col.num_entities} 条文本块已入库 Milvus")


### 6.14 Milvus 上的 RAG 问答

在线阶段标准三步：**检索（Retrieval）→ 增强（Augmented）→ 生成（Generation）**

1. 用户问题转成向量，在 Milvus 中检索最相关的 Top-2 文本块
2. `build_prompt` 把文本块作为上下文填入 Prompt 模板（含"无法回答"兜底指令）
3. 调用大模型生成带知识库依据的回答


In [ ]:
user_query = "Llama 2 有多少参数"

# ① 检索：问题向量化 → Milvus 召回 Top-2 文本块
qv = get_embeddings([user_query])
hits = rag_col.search(
    data=qv, anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"nprobe": 8}},
    limit=2, output_fields=["chunk_text"]
)
context = [h.entity.get("chunk_text") for h in hits[0]]

print("📄 检索到的上下文：")
for c in context:
    print("  -", c[:80], "...")

# ② 增强：检索结果填入 Prompt 模板
prompt = build_prompt(prompt_template, context=context, query=user_query)

# ③ 生成：调用大模型回答
print("\n🤖 RAG 回答：")
print(get_completion(prompt))


### 6.15 小结：向量数据库全景回顾

**一个核心流程**：原始数据 → 向量化 → 建索引 → 相似性检索 → 返回结果

**三大核心技术**：
1. **向量嵌入**：把万物翻译成数学语言（静态 Word2Vec / 动态 CLIP、BERT）
2. **相似度度量**：欧氏距离看"长短"，余弦相似度看"方向"（语义检索首选）
3. **向量索引**：IVF 先分桶再局部搜索；HNSW 分层小世界图，亿级数据毫秒级响应

**两类产品**：轻量工具库（FAISS）与 企业级数据库（Milvus）

**本课闭环**：`llama2.pdf → 切分 → Embedding → Milvus 检索 → Prompt 增强 → LLM 生成`——这就是生产级 RAG 的标准架构。

**思考题**：
1. 调整 `nlist`（16→128）与 `nprobe`（1→16），观察检索速度与召回率的变化
2. 把索引换成 `HNSW`，对比 IVF_FLAT 的效果差异
3. 从架构、容量、运维角度，比较 Chroma 与 Milvus 的差异？
